In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import torch
import umap

from collections import Counter
from matplotlib_venn import venn2 
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.metrics import (
	silhouette_score,
	davies_bouldin_score,
	calinski_harabasz_score,
	adjusted_rand_score,
	normalized_mutual_info_score,
	adjusted_mutual_info_score
)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [ ]:
# Linear models
from sklearn.linear_model import (
	LogisticRegression,
	RidgeClassifier,
	SGDClassifier,
	Perceptron,
	PassiveAggressiveClassifier
)

# Trees
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier

# Ensembles
from sklearn.ensemble import (
	RandomForestClassifier,
	ExtraTreesClassifier,
	BaggingClassifier,
	AdaBoostClassifier,
	GradientBoostingClassifier,
	HistGradientBoostingClassifier
)

# SVM
from sklearn.svm import SVC, NuSVC, LinearSVC

# KNN
from sklearn.neighbors import KNeighborsClassifier, RadiusNeighborsClassifier

# Neural network
from sklearn.neural_network import MLPClassifier

# Naive Bayes
from sklearn.naive_bayes import (
	GaussianNB,
	MultinomialNB,
	BernoulliNB,
	ComplementNB
)

# Discriminant Analysis
from sklearn.discriminant_analysis import (
	LinearDiscriminantAnalysis,
	QuadraticDiscriminantAnalysis
)

# Gaussian Process
from sklearn.gaussian_process import GaussianProcessClassifier

### Summary results analysis

In [ ]:
# print(exp)
exp = "exp110"
file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

exp = params["exp"]
print("Exp:\t\t", exp)

methods = params["methods"]
print("Methods:\t", methods)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

controls = params["controls"]
print("Control:\t", controls)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups = params["groups"]
print("Groups:\t\t", groups)

#### Preprocessing

In [ ]:
df_preprocessing = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp))
df_preprocessing
df_preprocessing["Node features"] = ["signal intensity (3)"] * len(df_preprocessing)
# print(df_preprocessing.to_latex(index=False, formatters={"name": str.upper}, float_format="{:.2f}".format)) 
df_preprocessing

In [ ]:
# Correlation plots
for group_id in groups_id:
	for subgroup_id in subgroups_id[group_id]:
		df_matrix = pd.read_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index_col=0)
		print(group_id, subgroup_id, df_matrix.to_numpy().min(), df_matrix.to_numpy().max())

In [ ]:
# Correlation plots
for group_id in groups_id:
	for subgroup_id in subgroups_id[group_id]:
		df_matrix = pd.read_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index_col=0)
		
		from matplotlib.colors import TwoSlopeNorm
		norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
		# plt.figure(figsize=(8, 6))
		img = plt.imshow(
			df_matrix,
			cmap="coolwarm",
			interpolation="none",
			norm=norm
		)
		plt.colorbar(img)
		plt.title(f"{group_id}, {subgroup_id}")
		plt.show()

#### Node-Edge embeddings (common subgraph)

In [ ]:
f = "f1" # change: f1, f2

df_process = pd.read_csv("experiments/output/{}/common_edges/summary_{}.csv".format(exp, f))
df_process
""" df_comparation_temp = df_process.set_index(["Method", "Group", "Data var."])
df_comparation_temp.sort_values(by=["Method", "Group", "Num. edges"], ascending=False, inplace=True)
ax = df_comparation_temp.plot.bar(rot=90)
ax.grid() """
df_join_raw = pd.read_csv("experiments/input/{}_raw.csv".format(exp), index_col=0, usecols=[0, 1, 2, 3])        
# df_join_raw.columns = ["mz", "name"]
df_join_raw
# details
list_details = []

for group in groups_id:
	df_common_edges = pd.read_csv("experiments/output/{}/common_edges/common_edges_{}_{}_{}.csv".format(exp, "greedy", group, "none"))
	G = nx.from_pandas_edgelist(df_common_edges, edge_attr=["weight"])
	nodes = list(G.nodes())
	df_nodes_raw = df_join_raw.loc[nodes]
	
	num_nodes_name = len(df_nodes_raw[df_nodes_raw["Metabolite name"] != "Unknown"])
	
	list_details.append(["greedy", group, "none", G.number_of_nodes(), G.number_of_edges(), nx.density(G), num_nodes_name])

for method in methods:
	for group in groups_id:
		for data_variation in data_variations:
			df_edges_filter_weight_filter = pd.read_csv("experiments/output/{}/common_edges/common_edges_{}_{}_{}_{}.csv".format(exp, f, method, group, data_variation))

			G = nx.from_pandas_edgelist(df_edges_filter_weight_filter, "source", "target", edge_attr="weight")
			nodes = list(G.nodes())
			df_nodes_raw = df_join_raw.loc[nodes]
			
			num_nodes_name = len(df_nodes_raw[df_nodes_raw["Metabolite name"] != "Unknown"])
			
			list_details.append([method, group, data_variation, G.number_of_nodes(), G.number_of_edges(), nx.density(G), num_nodes_name])

df_details = pd.DataFrame(list_details, columns=["Method", "Group", "Data var.", "Num. nodes", "Num. edges", "Density", "With name"])
df_details
# percentage

max_nodes = df_details.iloc[:3, :]["Num. nodes"].to_list()
max_nodes

max_edges = df_details.iloc[:3, :]["Num. edges"].to_list()
max_edges

total = []
nodes = []
edges = []

for i in tqdm(df_details.index[:len(groups_id)]):
	total.append([df_details["Num. nodes"][i], df_details["Num. edges"][i]])
	nodes.append("{} ({}%)".format(df_details["Num. nodes"][i], round(df_details["Num. nodes"][i]*100/df_details["Num. nodes"][i], 2)))
	edges.append("{} ({}%)".format(df_details["Num. edges"][i], round(df_details["Num. edges"][i]*100/df_details["Num. edges"][i], 2)))

index = 0
c = 0
for i in tqdm(df_details.index[len(groups_id):]):
	nodes.append("{} ({}%)".format(df_details["Num. nodes"][i], round(df_details["Num. nodes"][i]*100/total[index][0], 2)))
	edges.append("{} ({}%)".format(df_details["Num. edges"][i], round(df_details["Num. edges"][i]*100/total[index][1], 2)))

	c += 1
	if c % len(data_variations) == 0:
		index = (index + 1) % len(groups_id)
		
df_results_ = df_details.copy()
df_results_["Num. nodes"] = nodes
df_results_["Num. edges"] = edges
df_results_
df_results_.to_csv("z_percentage_mutan_random.csv", index=False)

#### Similarity analysis (change detection)

In [ ]:
df_changes = pd.read_csv("experiments/output/{}/changes/summary.csv".format(exp))
df_changes
df_join_raw = pd.read_csv("experiments/input/{}_raw.csv".format(exp), index_col=0, usecols=[0, 1, 2])        
# df_join_raw.columns = ["mz", "name"]
df_join_raw
# count metabolities by name in raw data
size1 = len(df_join_raw)
size2 = len(df_join_raw[df_join_raw["Metabolite name"] != "Unknown"])
print(size1, size2)
methods.insert(0, "greedy")
methods
# details

is_significant = False # change, True: significant changes (*), False: no significant changes (non-* and same labels)
list_details = []

for method in methods:
	if method == "greedy":
		data_variations = ["none"]
	else:
		data_variations = ["none", "str", "dyn"]
	for data_variation in data_variations:
		for group in groups:
			df_change_filter = pd.read_csv("experiments/output/{}/changes/changes_edges_log2_{}_{}_{}_{}.csv".format(exp, method, group[0], group[1], data_variation))
			
			if is_significant:
				df_change_filter = df_change_filter[df_change_filter["significant"] == "*"]
			else:
				df_change_filter = df_change_filter[df_change_filter["significant"] != "*"]
				df_change_filter = df_change_filter[df_change_filter["label"].isin(["nn", "NN", "pp", "PP"])]
			
			# df_change_filter = df_change_filter[df_change_filter["label"].str.contains("?", regex=False) == False] # change uncomment this line to view (intersection)
			
			G = nx.from_pandas_edgelist(df_change_filter, "source", "target", edge_attr="weight1")
			nodes = list(G.nodes())
			df_nodes_raw = df_join_raw.loc[nodes]
			
			num_nodes_name = len(df_nodes_raw[df_nodes_raw["Metabolite name"] != "Unknown"])
			
			list_details.append([method, "-".join(group), data_variation, G.number_of_nodes(), G.number_of_edges(), nx.density(G), num_nodes_name])

df_details = pd.DataFrame(list_details, columns=["Method", "Groups", "Data var.", "Num. nodes", "Num. edges", "Density", "With name"])
df_details
df_details.to_csv("z.csv", index=False)
# details count labels

is_significant = False # change, True: significant changes (*), False: no significant changes (non-* and same labels)

list_details = []
labels = ['PP', 'Pp', 'PN', 'Pn', 'P?', 'pP', 'pp', 'pN', 'pn', 'p?', 'NP', 'Np', 'NN', 'Nn', 'N?', 'nP', 'np', 'nN', 'nn', 'n?', '?P', '?p', '?N', '?n']

for method in methods:
	if method == "greedy":
		data_variations = ["none"]
	else:
		data_variations = ["none", "str", "dyn"]
	for data_variation in data_variations:
		for group in groups:
			df_change_filter = pd.read_csv("experiments/output/{}/changes/changes_edges_log2_{}_{}_{}_{}.csv".format(exp, method, group[0], group[1], data_variation))
			
			if is_significant:
				df_change_filter = df_change_filter[df_change_filter["significant"] == "*"]
			else:
				df_change_filter = df_change_filter[df_change_filter["significant"] != "*"]
				df_change_filter = df_change_filter[df_change_filter["label"].isin(["nn", "NN", "pp", "PP"])]
				
			counts = []
			for label in labels:
				counts.append(len(df_change_filter[df_change_filter.label == label]))
			
			list_details.append([method, "-".join(group), data_variation] + counts)

df_details = pd.DataFrame(list_details, columns=["Method", "Groups", "Data var."] + labels)
df_details.to_csv("z.csv", index=False)

### Plot node-embeddings

In [ ]:
exp = "exp110"
file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

exp = params["exp"]

print("Exp:\t\t", exp)

methods = params["methods"]
print("Methods:\t", methods)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

controls = params["controls"]
print("Control:\t", controls)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups = params["groups"]
print("Groups:\t\t", groups)

encoders = ["GIN"] # ["GIN", "GINE"]

In [ ]:
def plot_embedding_3d(df_embeddings, labels, reduction="pca", embedding="node", title="", title_legend="", save=False):
	# print(df_embeddings)
	# print(labels)
	if df_embeddings.shape[1] > 3:
		if reduction == "pca":
			df_embeddings_red = PCA(n_components=3).fit_transform(df_embeddings)
		elif reduction == "tsne":
			df_embeddings_red = TSNE(n_components=3).fit_transform(df_embeddings)
		elif reduction == "umap":
			# df_embeddings_red = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42).fit_transform(df_embeddings)
			# df_embeddings_red = umap.UMAP(n_components=3, init="random", random_state=42).fit_transform(df_embeddings)
			df_embeddings_red = umap.UMAP(n_components=3, init="pca", random_state=42).fit_transform(df_embeddings)
		elif reduction == "kpca":
			df_embeddings_red = KernelPCA(n_components=3, kernel="cosine", n_jobs=-1, random_state=42).fit_transform(df_embeddings)
	else:
		df_embeddings_red = df_embeddings.values
	
	fig = plt.figure(figsize=(6, 6))
	ax = fig.add_subplot(projection="3d")
	
	df_embeddings = pd.DataFrame(df_embeddings_red)
	df_embeddings["labels"] = labels.values
	
	unique_labels = np.unique(labels)
	for i, label in enumerate(unique_labels):
		df_embeddings_filter = df_embeddings[df_embeddings["labels"] == label]

		x = df_embeddings_filter.iloc[:, 0]
		y = df_embeddings_filter.iloc[:, 1]
		z = df_embeddings_filter.iloc[:, 2]
		
		# new_cmap = matplotlib.colors.ListedColormap(plt.cm.tab10.colors[i: i + 1])
		if embedding == "node":
			color = matplotlib.colors.ListedColormap(plt.cm.Dark2.colors[i: i + 1]).colors[0]
			# points = ax.scatter(x, y, z, s=100, c=labels, alpha=0.5, cmap=new_cmap , marker=".") # , edgecolors="black", linewidth=0.5)
			points = ax.scatter(x, y, z, s=100, alpha=0.5, color=color, marker=".", label="Br{}".format(label + 1))
		elif embedding == "edge":
			color = matplotlib.colors.ListedColormap(plt.cm.Dark2.colors[i: i + 1]).colors[0]
			# points = ax.scatter(x, y, z, s=100, c=labels, alpha=0.5, cmap=new_cmap , marker=".") # , edgecolors="black", linewidth=0.5)
			points = ax.scatter(x, y, z, s=100, alpha=0.5, color=color, marker=".", label="Br{}".format(label + 1))
		elif embedding == "outlier":
			# cmap = matplotlib.colormaps.get_cmap("coolwarm", len(unique_labels))
			colors = plt.get_cmap("coolwarm")(np.linspace(0, 1, len(unique_labels)))
			points = ax.scatter(x, y, z, s=100, alpha=0.5, c=[colors[label]], marker=".", label=["inliers", "outliers"][label]) #, edgecolors="black", linewidth=0.5)
	
	# ax.set_xlabel("X")
	# ax.set_ylabel("Y")
	# ax.set_zlabel("Z")
	if not save:
		""" plt.title(title)
		plt.legend(title=title_legend, ncol=1, loc=0, bbox_to_anchor=(1, 0.95)) """
		# fig.colorbar(points, ax=ax, shrink=0.4, aspect=8) # bar
		# ax.legend(["Biological rep.: {}".format(subgroup) for subgroup in np.unique(labels)])
		# print(*points.legend_elements())
		# plt.legend(*points.legend_elements(), bbox_to_anchor=(1.05, 1), loc=2)
	if save:
		# splt.legend(title=title_legend, ncol=1, loc=0, bbox_to_anchor=(1, 0.95)) # ncol=len(unique_labels)
		plt.savefig("experiments/output/{}.pdf".format(title), format="pdf", bbox_inches="tight") # change    
	plt.show()
	return ax

def mu(X):
	# X = X.detach()
	centered = X - torch.mean(X, dim = -2)
	return torch.linalg.norm(centered).item()

In [ ]:
map_group = {
	"FrescoAmazonas": "FreshAmazonas",
	"FrescoCusco": "FreshCusco",
	"FrescoSanMartin": "FreshSanMartín",
	"SecoAmazonas": "DryAmazonas",
	"SecoCusco": "DryCusco",
	"SecoSanMartin": "DrySanMartín",
}

In [ ]:
# Remove
# groups_id = ["LSNB", "BC", "RCC", "BPH", "PD"] # "OSA" Memory issue

2D

In [ ]:
# Node embeddings 2D
save = False

for encoder in encoders:
	for group_id in groups_id:
		print(encoder, group_id)

		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv")
		# Id, 0, 1,	2, ..., subgroup_id

		node_embeddings_concat = df_node_embeddings_concat.iloc[:, 1:-1].values
		labels = df_node_embeddings_concat.iloc[:, -1].values

		if node_embeddings_concat.shape[1] > 2:
			pca = PCA(n_components=2)
			node_embeddings_concat_2d = pca.fit_transform(node_embeddings_concat)
		else:
			node_embeddings_concat_2d = node_embeddings_concat.copy()

		fig, ax = plt.subplots()

		for c in np.unique(labels):
			idx = labels == c
			ax.scatter(
				node_embeddings_concat_2d[idx, 0],
				node_embeddings_concat_2d[idx, 1],
				s=10,
				alpha=0.8,
				label=f"{group_id} {c}" # change
				# label=f"{map_group[group_id]} {c}" # change
			)

		ax.set_xlabel("X")
		ax.set_ylabel("Y")
		ax.legend()
		# plt.axis("equal")
		plt.tight_layout()
		if save:
			plt.savefig(f"experiments/output/{exp}/plots/node_embeddings_2d_{encoder}_{group_id}.pdf", format="pdf", bbox_inches="tight")
		plt.show()

3D

In [ ]:
# Node embeddings 3D
save = True

for encoder in encoders:
	for group_id in groups_id:
		print(encoder, group_id)

		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv")
		# Id, 0, 1,	2, ..., subgroup_id

		node_embeddings_concat = df_node_embeddings_concat.iloc[:, 1:-1].values
		labels = df_node_embeddings_concat.iloc[:, -1].values

		if node_embeddings_concat.shape[1] > 3:
			pca = PCA(n_components=3)
			node_embeddings_concat_3d = pca.fit_transform(node_embeddings_concat)
		else:
			node_embeddings_concat_3d = node_embeddings_concat.copy()

		fig = plt.figure()
		ax = fig.add_subplot(projection="3d")

		for c in np.unique(labels):
			ax.scatter(
				node_embeddings_concat_3d[:, 0][labels == c],
				node_embeddings_concat_3d[:, 1][labels == c], 
				node_embeddings_concat_3d[:, 2][labels == c],
				s=10,
				alpha=0.8,
				label=f"{group_id} {c}" # change
				# label=f"{map_group[group_id]} {c}" # change
			)

		ax.set_xlabel("X")
		ax.set_ylabel("Y")
		ax.set_zlabel("Z")
		ax.legend()
		# plt.axis("equal")
		plt.tight_layout()
		if save:
			plt.savefig(f"experiments/output/{exp}/plots/node_embeddings_3d_{encoder}_{group_id}.pdf", format="pdf", bbox_inches="tight")
		plt.show()

In [ ]:
iteration = 1 # change
save = False # change

for encoder in encoders:
	for exp_ in [exp]: # change, experiment
		# exp = "exp" + str(e)
		for method in methods[:]: # change
			for group_id in groups_id[:]: # change
					df_node_embeddings_concat = pd.read_csv("experiments/output/{}/node_embeddings/{}_{}.csv".format(exp_, encoder, group_id))
					print(method, encoder, group_id)
					print(mu(torch.tensor(df_node_embeddings_concat.iloc[:, 1:-1].values)))
					plot_embedding_3d(df_node_embeddings_concat.iloc[:, 1:-1], df_node_embeddings_concat.iloc[:, -1],
										reduction="pca", embedding="node", title="{}/plots/node-embeddings_{}".format(exp_, group_id), title_legend="", save=save)

					# plot_embeddings_3d_plotly(df_node_embeddings_concat.iloc[:, 1:-1], df_node_embeddings_concat.iloc[:, -1], df_node_embeddings_concat.iloc[:, 0], reduction="pca")

### Plots

In [ ]:
exp = "exp121"
file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

exp = params["exp"]

print("Exp:\t\t", exp)

methods = params["methods"]
print("Methods:\t", methods)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

controls = params["controls"]
print("Control:\t", controls)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups = params["groups"]
print("Groups:\t\t", groups)

# encoders = ["GIN", "GINE"]
encoders = ["GCNRes", "GIN", "GINE"] # ["GCNRes", "GCN", "GIN", "GINE"]

In [ ]:
# groups_id = ['SecoAmazonas', 'SecoCusco', 'SecoSanMartin', 'FrescoAmazonas', 'FrescoCusco', 'FrescoSanMartin']

In [ ]:
# Only for Pabloś dataset

# subgroups_id = ["5", "6"]
# subgroups_id = ["5", "6", "8"]
# subgroups_id = ["1", "13", "17"]
# subgroups_id = ["2", "14", "16"]
# subgroups_id = ["1", "9", "13", "17"]
subgroups_id = ["11", "12"]
# subgroups_id = ["10", "18"]


"-".join(subgroups_id)

In [ ]:
map_group = {
	"FrescoAmazonas": "FreshAmazonas",
	"FrescoCusco": "FreshCusco",
	"FrescoSanMartin": "FreshSanMartín",
	"SecoAmazonas": "DryAmazonas",
	"SecoCusco": "DryCusco",
	"SecoSanMartin": "DrySanMartín",
}

#### Loss

In [ ]:
# Loss
save = False # Change
for encoder in encoders:
	for group_id in groups_id:
		print(encoder, group_id)
		list_loss = np.load(f"experiments/output/{exp}/loss/{encoder}_{group_id}.npy") # This is original
		""" aux = "-".join(subgroups_id)
		list_loss = np.load(f"experiments/output/{exp}/loss/{encoder}_{group_id}_{aux}.npy") """

		plt.figure()
		plt.plot(range(1, len(list_loss) + 1), list_loss) #, marker=".")
		# plt.plot(range(1, len(list_loss) + 1), np.log(list_loss)) #, marker=".")
		# plt.title("Training Loss over Epochs")
		plt.xlabel("Epochs")
		plt.ylabel("Loss")
		plt.grid()
		plt.tight_layout()
		if save:
			plt.savefig(f"experiments/output/{exp}/plots/loss_{encoder}_{group_id}.pdf", format="pdf", bbox_inches="tight")
		plt.show()

#### Node embeddings

In [ ]:
# Node embeddings 3D

save = False # Change
for encoder in encoders:
	for group_id in groups_id:
		print(encoder, group_id)

		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv") # This is original
		""" aux = "-".join(subgroups_id)
		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}_{aux}.csv") """
		# Id, 0, 1,	2, ..., subgroup_id

		node_embeddings_concat = df_node_embeddings_concat.iloc[:, 1:-1].values
		labels = df_node_embeddings_concat.iloc[:, -1].values

		if node_embeddings_concat.shape[1] > 3:
			pca = PCA(n_components=3)
			node_embeddings_concat_3d = pca.fit_transform(node_embeddings_concat)
		else:
			node_embeddings_concat_3d = node_embeddings_concat.copy()

		fig = plt.figure()
		ax = fig.add_subplot(projection="3d")

		for c in np.unique(labels):
			ax.scatter(
				node_embeddings_concat_3d[:, 0][labels == c],
				node_embeddings_concat_3d[:, 1][labels == c], 
				node_embeddings_concat_3d[:, 2][labels == c],
				s=10,
				alpha=0.3,
				label=f"{group_id} {c}"
			)

		ax.set_xlabel("X")
		ax.set_ylabel("Y")
		ax.set_zlabel("Z")
		ax.legend()
		plt.tight_layout()
		if save:
			plt.savefig(f"experiments/output/{exp}/plots/node_embeddings_3d_{encoder}_{group_id}.pdf", format="pdf", bbox_inches="tight")
		plt.show()

In [ ]:
# Node embeddings 2D

save = False # Change

for encoder in encoders:
	for group_id in groups_id:
		print(encoder, group_id)

		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv") # This is original
		# aux = "-".join(subgroups_id)
		# df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}_{aux}.csv")
		# Id, 0, 1,	2, ..., subgroup_id

		node_embeddings_concat = df_node_embeddings_concat.iloc[:, 1:-1].values
		labels = df_node_embeddings_concat.iloc[:, -1].values

		if node_embeddings_concat.shape[1] > 2:
			pca = PCA(n_components=2)
			node_embeddings_concat_2d = pca.fit_transform(node_embeddings_concat)
		else:
			node_embeddings_concat_2d = node_embeddings_concat.copy()

		fig, ax = plt.subplots()

		for c in np.unique(labels):
			idx = labels == c
			ax.scatter(
				node_embeddings_concat_2d[idx, 0],
				node_embeddings_concat_2d[idx, 1],
				s=10,
				alpha=0.3,
				# label=f"{group_id} {c}"
				label=f"{map_group[group_id]} {c}"
			)

		ax.set_xlabel("X")
		ax.set_ylabel("Y")
		ax.legend()
		plt.tight_layout()
		if save:
			plt.savefig(f"experiments/output/{exp}/plots/node_embeddings_2d_{encoder}_{group_id}.pdf", format="pdf", bbox_inches="tight")
		plt.show()

### Similarity analysis (KNN) (only for MS dataset)

In [ ]:
exp = "exp100"
file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

exp = params["exp"]

print("Exp:\t\t", exp)

methods = params["methods"]
print("Methods:\t", methods)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

controls = params["controls"]
print("Control:\t", controls)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups = params["groups"]
print("Groups:\t\t", groups)

# encoders = ["GIN", "GINE"]
encoders = ["GINE"]

In [ ]:
k = 1 # Change
list_count_meta = []

for encoder in encoders:
	for group_id in groups_id:
		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
		# Id, 0, 1,	2, ..., subgroup_id

		subgroups_id_ = subgroups_id[group_id]

		# Calculate distance matrix (KNN)
		knn = NearestNeighbors(n_neighbors=k, metric="euclidean")

		df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroups_id_[0]]
		x = df_node_embeddings.iloc[:, 1:-1].values

		df_node_alignment = pd.DataFrame()
		df_node_alignment[f"{group_id}_{subgroups_id_[0]}"] = df_node_embeddings["Id"].values

		for subgroup_id in subgroups_id_[1:]:
			df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
			y = df_node_embeddings.iloc[:, 1:-1].values

			knn.fit(y)
			distances, indices = knn.kneighbors(x)
			indices = indices.squeeze() # (N,)

			df_node_alignment[f"{group_id}_{subgroup_id}"] = df_node_embeddings["Id"].values[indices]
			
			""" df_node_alignment[f"distances"] = distances
			avg_distances = df_node_alignment["distances"].mean()
			df_node_alignment = df_node_alignment[df_node_alignment["distances"] <= avg_distances].iloc[:, :-1]
			df_node_alignment """
		
		# Save previus node alignment
		df_node_alignment.to_csv(f"experiments/output/{exp}/filter_raw/node_alignment_{encoder}_{group_id}.csv")
		display(df_node_alignment)

		# Find node alignment for all datasets
		""" df_node_alignment_filter = df_node_alignment[df_node_alignment.nunique(axis=1) == 1]
		list_count_meta.append([encoder, group_id, len(df_node_alignment_filter), len(df_node_alignment)])
		# print(len(df_node_alignment_filter))
		# display(df_node_alignment_filter)
		print(f"{encoder}-{group_id}: {len(df_node_alignment_filter)}")
		
		# Save common node id
		# common_node_id = df_node_alignment_filter.iloc[:, 0].values
		# np.save(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy", np.array(common_node_id)) """

		# Save common node id as .csv
		df_common_node_id = df_node_alignment.iloc[:, [0, 1]].copy() # df_common_node_id = df_node_alignment_filter.iloc[:, [0, 1]].copy()
		df_common_node_id.sort_values(by=df_common_node_id.columns[0], inplace=True)
		df_common_node_id.to_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", index=False, header=False)

# Intersection and Union
""" list_common_node_id = []
for encoder in encoders:
	for group_id in groups_id:
		# Read common node
		common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
		list_common_node_id.append(common_node_id)

	common_node_id_union = list(set.union(*map(set, list_common_node_id)))
	common_node_id_intersection = list(set.intersection(*map(set, list_common_node_id)))

	# Save common node id
	np.save(f"experiments/output/{exp}/common_nodes/{encoder}_union.npy", np.array(common_node_id_union))
	np.save(f"experiments/output/{exp}/common_nodes/{encoder}_intersection.npy", np.array(common_node_id_intersection)) """

In [ ]:
# Get common node id
for encoder in encoders:
	for group_id in groups_id + ["union", "intersection"]:
		# Read common node
		common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
		print(group_id, common_node_id)

In [ ]:
# Comparison
list_count_meta

In [ ]:
list_count_meta_ = np.array(list_count_meta)

ig, ax = plt.subplots(figsize=(8, 5))

# Barras
bars_total = ax.bar(
	list_count_meta_[:, 1],
	list_count_meta_[:, 3].astype(int),
	label="Total nodes"
)

bars_common = ax.bar(
	list_count_meta_[:, 1],
	list_count_meta_[:, 2].astype(int),
	label="Common nodes"
)

# Mostrar valores sobre cada barra
ax.bar_label(bars_total, fmt="%d", padding=3, fontsize=9)
ax.bar_label(bars_common, fmt="%d", padding=3, fontsize=9, color="white")

# Etiquetas
ax.set_xlabel("Phenotypes")
ax.set_ylabel("Number of Node Alignments")

# Leyenda
ax.legend(loc="best")

# Cuadrícula
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
gin = []
gine = []
for k in range(int(len(list_count_meta)/2)):
	gin.append(list_count_meta[k][2])
	gine.append(list_count_meta[k+ int(len(list_count_meta)/2)][2])
print(gin)
print(gine)

In [ ]:
# file: plots/model_accuracy_comparison.py

import numpy as np
import matplotlib.pyplot as plt

save = True  # change

colors = [
	"#A1C9F4",
	"#FFB482",
	"#8DE5A1",
	"#FF9F9B",
]

# Datasets
datasets = [
	"DryAmazonas",
	"DryCusco",
	"DrySanMartin",
	"FreshAmazonas",
	"FreshCusco",
	"FreshSanMartin",
]

# Accuracy values
results = {
	"GIN": gin,
	"GINE": gine,
}

# Plot settings
x = np.arange(len(datasets))
width = 0.35
n_models = len(results)

fig, ax = plt.subplots(figsize=(10, 5))

# Draw bars
for i, (model, values) in enumerate(results.items()):

	# Center bars around each dataset position
	offset = (i - (n_models - 1) / 2) * width

	bars = ax.bar(
		x + offset,
		values,
		width,
		label=model,
		color=colors[i],
	)

	ax.bar_label(
		bars,
		padding=3,
		fontsize=9,
	)

# Axis labels
ax.set_xlabel("Dataset")
ax.set_ylabel("Number of Node Alignments")

# Dataset labels remain centered automatically
ax.set_xticks(x)
ax.set_xticklabels(
	datasets,
	# rotation=45,
	ha="center",
)

ax.set_ylim(0, list_count_meta[0][3])

# Legend
ax.legend(
	title="Encoder",
	ncols=n_models,
)

# Grid
ax.grid(
	axis="y",
	linestyle="--",
	alpha=0.5,
)

plt.tight_layout()

# Export figure
if save:
	plt.savefig(
		f"experiments/output/{exp}/plots/number_node_alignment.pdf",
		format="pdf",
		bbox_inches="tight",
	)

plt.show()

### Filter MS data

In [ ]:
exp = "exp8"
file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

exp = params["exp"]

print("Exp:\t\t", exp)

methods = params["methods"]
print("Methods:\t", methods)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

controls = params["controls"]
print("Control:\t", controls)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups = params["groups"]
print("Groups:\t\t", groups)

# encoders = ["GIN", "GINE"]
encoders = ["GIN"]

In [ ]:
# Read raw data

df_join_raw = pd.read_csv(f"experiments/input/{exp}_raw.csv", index_col=0)
df_join_raw

In [ ]:
# Filter
for encoder in encoders:
	for group_id in groups_id:
		# Read common node
		common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
		print(f"{encoder}-{group_id}: {len(common_node_id)}/{len(df_join_raw)}: {common_node_id}")

		# Filter raw data
		df_join_raw_filter = df_join_raw.loc[common_node_id].iloc[:, [0, 1, 2]]
		df_join_raw_filter.to_csv(f"experiments/output/{exp}/filter_raw/{encoder}_{group_id}.csv", sep=";", decimal=",", index_label="Id")
		df_join_raw_filter

In [ ]:
groups_id

In [ ]:
# Get node alignment
encoder = "GINE"
group_id = groups_id[4]
print(encoder, group_id)


common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
print(f"{encoder}-{group_id}: {len(common_node_id)}/{len(df_join_raw)}")

common_node_id = [2, 5, 15, 30,  32,  38, 46, 531, 532, 537, 538, 542, 551]

# Filter raw data
df_join_raw_filter = df_join_raw.loc[common_node_id].iloc[:, [0, 1, 2]]
df_join_raw_filter.to_csv(f"experiments/output/{exp}/filter_raw/{encoder}_{group_id}.csv", sep=";", decimal=",", index_label="Id")
df_join_raw_filter

In [ ]:
# Filter without NAN

df_join_raw_filter[df_join_raw_filter["Metabolite name"] != "Unknown"]

### Clustering analysis

In [ ]:
exp = "exp117"
file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

exp = params["exp"]

print("Exp:\t\t", exp)

methods = params["methods"]
print("Methods:\t", methods)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

controls = params["controls"]
print("Control:\t", controls)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups = params["groups"]
print("Groups:\t\t", groups)

# encoders = ["GIN", "GINE"]
encoders = ["GCNRes", "GCN", "GIN", "GINE"]

In [ ]:
# Read raw data

df_join_raw = pd.read_csv(f"experiments/input/{exp}_raw.csv", index_col=0)
df_join_raw

In [ ]:
map_group = {
	"FrescoAmazonas": "FreshAmazonas",
	"FrescoCusco": "FreshCusco",
	"FrescoSanMartin": "FreshSanMartín",
	"SecoAmazonas": "DryAmazonas",
	"SecoCusco": "DryCusco",
	"SecoSanMartin": "DrySanMartín",
}

In [ ]:
groups_id

In [ ]:
list_scores = []
reduction = "PCA" # Change PCA, TSNE, UMAP
save = False # Change

all_flag = True
vanessa_flag = True

retrievals = ["", "_cosine", "_l2"]
for encoder in encoders:
	for group_id in ["All", "vanessa", "union", "intersection"] + groups_id: # Important None for all nodes, others for common nodes
		for retrieval in retrievals:
			print()
			try:
				if group_id == "All" and all_flag:
					df_join_raw_signal = df_join_raw.iloc[:, 3:] # Important two last column no, only to Mentos 3:-2
					all_flag = False
					encoder = ""
					retrieval = ""
				elif group_id == "vanessa" and vanessa_flag:
					common_node_id = pd.read_csv(f"experiments/output/{exp}/common_nodes/vanessa.csv", header=None)
					common_node_id = common_node_id[0].to_numpy()
					
					df_join_raw_signal = df_join_raw.loc[common_node_id].iloc[:, 3:] # Important two last column no, only to Mentos 3:-2
					vanessa_flag = False
					encoder = ""
					retrieval = ""
				else:
					# Read common node
					# common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
					common_node_id = pd.read_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}{retrieval}.csv", header=None)
					common_node_id = common_node_id[0].to_numpy()
					# print(common_node_id)
					# print(group_id, len(common_node_id), common_node_id)
					# common_node_id = common_node_id[common_node_id != 20]
					df_join_raw_signal = df_join_raw.loc[common_node_id].iloc[:, 3:] # Important two last column no, only to Mentos 3:-2
				
				print(encoder, group_id, retrieval)

				df_join_raw_signal_t = df_join_raw_signal.T
				
				# Scaling
				scaler = StandardScaler()
				X_scaled = scaler.fit_transform(df_join_raw_signal_t.values)

				# Dim. reduction
				if reduction == "TSNE":
					X_reduction = TSNE(n_components=2, learning_rate="auto", init="pca", perplexity=3, random_state=42).fit_transform(X_scaled) # perplexity=5,
				elif reduction == "UMAP" :
					X_reduction = umap.UMAP(n_components=2, random_state=42).fit_transform(X_scaled) # metric="cosine",
				else:
					X_reduction = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
				
				# Plot
				labels = [item.split("_")[0] for item in df_join_raw_signal_t.index]
				
				# Silhouette score
				silhouette_val = silhouette_score(X_reduction, labels) # Cuanto más alto, mejor; rango de −1 a 1.
				davies_bouldin_val = davies_bouldin_score(X_reduction, labels) # Cuanto más bajo, mejor.
				calinski_harabasz_val = calinski_harabasz_score(X_reduction, labels) # Generalmente, cuanto más alto, mejor.

				list_scores.append([encoder, group_id, retrieval, silhouette_val, davies_bouldin_val, calinski_harabasz_val])
				# Plot
    
				""" x, y = X_reduction[:, 0], X_reduction[:, 1]
				unique_groups = np.unique(labels)
				# print(unique_groups, labels)

				for group in unique_groups:
					xi = [x[i] for i in range(len(x)) if labels[i] == group]
					yi = [y[i] for i in range(len(y)) if labels[i] == group]
					# plt.scatter(xi, yi, s=50, alpha=0.8, label=map_group[group])
					plt.scatter(xi, yi, s=50, alpha=0.8, label=group)

					# Only for delete outlier in exp5
					# for k in range(len(xi)):
					# 	plt.annotate(common_node_id[k], (xi[k], yi[k]))
				
				plt.axis("equal")
				plt.legend(fontsize="small", ncol=2)
				plt.xlabel("C1")
				plt.ylabel("C2")
				# plt.title(f"{group_id}")
				plt.tight_layout()
				if save:
					plt.savefig(f"experiments/output/{exp}/plots/clustering_{encoder}_{group_id}.pdf", format="pdf", bbox_inches="tight")
				plt.show() """
			except:
				# print("Error")
				pass
list_scores

In [ ]:
df_list_scores = pd.DataFrame(list_scores)
df_list_scores.columns = ["Encoder", "Group", "Retrieval", "Silhouette ↑", "Davies-Bouldin ↓", "Calinski-Harabasz ↑"]
df_list_scores.sort_values(by=["Silhouette ↑"], inplace=True, ascending=False)
df_list_scores

In [ ]:
df_list_scores[50:]

In [ ]:
df_list_scores.sort_values(by=["Davies-Bouldin ↓"], inplace=True)
df_list_scores

In [ ]:
df_list_scores.sort_values(by=["Calinski-Harabasz ↑"], inplace=True, ascending=False)
df_list_scores

### Network alignment (metrics)

In [ ]:
exp = "exp100"
file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

exp = params["exp"]

print("Exp:\t\t", exp)

methods = params["methods"]
print("Methods:\t", methods)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

controls = params["controls"]
print("Control:\t", controls)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups = params["groups"]
print("Groups:\t\t", groups)

# encoders = ["GIN", "GINE"]
encoders = ["GIN"]

### Comparison Br, Ar

In [ ]:
# row_data_names = ["Pablo_2br_4ar_p_42_format", "Pablo_2br_4ar_p_43_format", "Pablo_2br_4ar_p_44_format"]
# exps = ["exp15", "exp16", "exp17"]
# exps = ["exp18", "exp19", "exp20"]
# exps = ["exp21", "exp22", "exp23"]
# exps = ["exp24", "exp25", "exp26"]
# exps = ["exp27", "exp28" , "exp29"]
# exps = ["exp30", "exp31" , "exp32"]
# exps = ["exp33", "exp34" , "exp35"]
exps = ["exp36", "exp37"]

# groups_id = ["AR", "Health", "CKD-Mild", "CKD-Severe", "LPRD"]
groups_id = ["disease"]

encoders = ["GIN"]

In [ ]:
for encoder in encoders:
	for group_id in groups_id:
		print(group_id)
		list_common_node_id = []
		for exp in exps:
			file = open("experiments/output/{}/parameters.json".format(exp))
			params = json.load(file)
			raw_data_file = params["raw_data_file"]

			# Read common node
			# common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
			common_node_id = pd.read_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", header=None)
			list_common_node_id.append(common_node_id[0].to_numpy())
			# print(common_node_id[0].to_numpy())
			print(f"{exp}, {raw_data_file}, {encoder}, {group_id}: {len(common_node_id)}")
	
		# common_node_id_union = list(set.union(*map(set, list_common_node_id)))
		common_node_id_intersection = list(set.intersection(*map(set, list_common_node_id)))
		# print(f"{encoder}-union: {len(common_node_id_union)}")
		print(f"{encoder}-intersection: {len(common_node_id_intersection)}")


### Networks analysis (Pablo)

In [ ]:
!pip install jupyter

In [ ]:
# !python -m pip install algorithmx

In [ ]:
# !pip install algorithmx
# !python -m jupyter nbextension enable --sys-prefix --py algorithmx

In [ ]:
exp = "exp38"
file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

exp = params["exp"]

print("Exp:\t\t", exp)

methods = params["methods"]
print("Methods:\t", methods)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

controls = params["controls"]
print("Control:\t", controls)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups = params["groups"]
print("Groups:\t\t", groups)

# encoders = ["GIN", "GINE"]
encoders = ["GIN"]

In [ ]:
# Read common nodes
# Read raw data

df_join_raw = pd.read_csv(f"experiments/input/{exp}_raw.csv", index_col=0)
df_join_raw

In [ ]:
# Only for Pabloś dataset

# subgroups_id = ["5", "6"]
# subgroups_id = ["5", "6", "8"]
# subgroups_id = ["1", "13", "17"]
# subgroups_id = ["2", "14", "16"]
# subgroups_id = ["1", "9", "13", "17"]
# subgroups_id = ["11", "12"]
# subgroups_id = ["10", "18"]

subgroups_id_ = subgroups_id[groups_id[0]]
subgroups_id_

In [ ]:
# Filter
for encoder in encoders:
	for group_id in groups_id:
		# Read common node
		aux = "-".join(subgroups_id_)
		common_node_id = pd.read_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}_{aux}.csv", header=None)
		print(f"{encoder}-{group_id}: {len(common_node_id)}/{len(df_join_raw)}: {common_node_id}")

		# Filter raw data
		df_join_raw_filter = df_join_raw.loc[common_node_id[0].values].iloc[:, :]
		# df_join_raw_filter.to_csv(f"experiments/output/{exp}/filter_raw/{encoder}_{group_id}.csv", sep=";", decimal=",", index_label="Id")
		df_join_raw_filter

In [ ]:
df_join_raw_filter

In [ ]:
group_id

In [ ]:
subgroups_id_

In [ ]:
# Filter raw data
# df_join_raw_filter = df_join_raw.loc[common_node_id].iloc[:, [0, 1, 2]]
df_join_raw_filter.to_csv(f"experiments/output/{exp}/filter_raw/{encoder}_{group_id}_{'-'.join(subgroups_id_)}.csv", sep=";", decimal=",", index_label="Id")
df_join_raw_filter

In [ ]:
groups_id

In [ ]:
subgroups_id

In [ ]:
# Partial correlation

from utils_MS import *

dict_df_groups_subgroups = split_groups_subgroups(df_join_raw_filter, groups_id, subgroups_id)
dict_df_groups_subgroups

In [ ]:
# Aplly Transpose

dict_groups_subgroups_t = transpose_global(dict_df_groups_subgroups)
dict_groups_subgroups_t

In [ ]:
dict_groups_subgroups_t_corr = correlation_global(exp, dict_groups_subgroups_t)
dict_groups_subgroups_t_corr

In [ ]:
# Build graph (corpus graphs)
threshold_corr = 0.5
dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g

In [ ]:
dict_groups_subgroups_t_corr_g.keys()

In [ ]:
# Get edges
list_edges = []

for subgroup_id in subgroups_id_:
	df_edges = dict_groups_subgroups_t_corr_g["disease"][subgroup_id]
	list_edges.append(df_edges)

In [ ]:
def graph_information(G):
	print("Number of nodes:", G.number_of_nodes())
	print("Number of edges:", G.number_of_edges())
	print("Nodes:", G.nodes())
	print("Nodes:", G.nodes(data=True))
	print("Edges:", G.edges())
	print("Edges:", G.edges(data=True))
	print("Is directed:", G.is_directed())
	print("Is weighted:", nx.is_weighted(G))
	
def obtener_label(peso):
	"""Devuelve el label según los rangos de peso especificados (incluyendo negativos)."""
	# Rangos para valores Positivos
	if 0.5 <= peso <= 0.7:
		return 'p'
	elif 0.7 < peso <= 1.0:
		return 'P'
	
	# Rangos para valores Negativos
	elif -0.7 <= peso <= -0.5:
		return 'n'
	elif -1.0 <= peso < -0.7:
		return 'N'
	
	return '*' # Por si algún peso cae fuera de estos rangos

def asignar_labels_a_grafo(G):
	"""Recorre las aristas del grafo y les asigna el atributo 'label' basado en su 'weight'."""
	for u, v, data in G.edges(data=True):
		peso = data.get('weight', 0)
		# Asignamos el nuevo atributo 'label' a la arista
		G[u][v]['label'] = obtener_label(peso)

In [ ]:
# Create graphs

list_graphs = []

for df_edges in list_edges:
	G = nx.from_pandas_edgelist(df_edges, source="source", target="target", edge_attr="weight")
	list_graphs.append(G)
	graph_information(G)
	
	# Convertir y asignar los labels en cada grafo antes de unirlos
	asignar_labels_a_grafo(G)
	graph_information(G)

In [ ]:
break

In [ ]:
# Union G1 and G2 (improve from here)
G_final = G1.copy()

for u, v, d in G_final.edges(data=True):
	d['label'] = obtener_label(d.get('weight', 0))

# 3. FUSIÓN ÓPTIMA: El método más rápido en NetworkX para combinar con condiciones
for u, v, d2 in G2.edges(data=True):
	peso_g2 = d2.get('weight', 0)
	label_g2 = obtener_label(peso_g2)
	
	# Intentamos obtener los datos de la arista en G_final de forma directa
	d1 = G_final.get_edge_data(u, v)
	
	if d1 is not None:
		# Si la arista ya existe, mutamos sus atributos in-place (velocidad máxima)
		d1['label'] += label_g2
		d1['weight'] = (d1['weight'] + peso_g2) / 2
	else:
		# Si no existe, la insertamos limpia
		G_final.add_edge(u, v, weight=peso_g2, label=label_g2)

graph_information(G_final)

In [ ]:
import algorithmx
canvas = algorithmx.jupyter_canvas()

# Añadir todos los nodos mapeados desde G_final
canvas.nodes(G_final.nodes).add()

# Añadir las aristas y configurar su texto usando una función lambda
# que busque la propiedad 'label' que creamos en cada arista 'e'
canvas.edges(G_final.edges).add() \
	.label().add() \
	.text(lambda e: G_final.edges[e].get('label', ''))

# Mostrar la visualización en pantalla
canvas

In [ ]:
# Edge list

df_grafo = pd.DataFrame([
	{
		'source': u, 
		'target': v, 
		'label': d.get('label', '-')
	} 
	for u, v, d in G_final.edges(data=True)
])
df_grafo

In [ ]:
subgroups_id_

In [ ]:
# Save graph similarity
df_grafo.to_csv(f"experiments/output/{exp}/filter_raw/{encoder}_{group_id}_gaph_{'-'.join(subgroups_id_)}.csv", index=None)


In [ ]:
np.unique(df_grafo[["label"]])

### Comparison MS data v1

In [ ]:
dataset1 = "deybis_05" # "deybis_05", "vanessa_05", "mentos_05", "Douban Online_Offline", "ACM_DBLP" # args.dataset
dataset2 = "vanessa_05" # "deybis_05", "vanessa_05", "mentos_05", "Douban Online_Offline", "ACM_DBLP" # args.dataset

encoder = "GIN"

In [ ]:
list_train_set = [
	["FrescoAmazonas_1", "FrescoAmazonas_2"],
	["FrescoCusco_1", "FrescoCusco_2"],
	["FrescoSanMartin_1", "FrescoSanMartin_2"],
	["SecoAmazonas_1", "SecoAmazonas_2"],
	["SecoCusco_1", "SecoCusco_2"],
	["SecoSanMartin_1", "SecoSanMartin_2"]
]

In [ ]:
tol_rt, tol_mz = 0.5, 15 # 0.5 min, 10 ppm

for train_set in list_train_set:
	df_1 = pd.read_csv(f"data/{dataset1}/output/node_alignment_{encoder}_{dataset1}_{train_set}.csv" , sep=";", decimal=",")
	df_2 = pd.read_csv(f"data/{dataset2}/output/node_alignment_{encoder}_{dataset2}_{train_set}.csv" , sep=";", decimal=",")

	df_match = (
		df_1.assign(key=1)
			.merge(df_2.assign(key=1), on="key", suffixes=("_1", "_2"))
			.drop(columns="key")
	)

	df_match["mz_error"] = (
		(df_match["Average Mz_1"] - df_match["Average Mz_2"]).abs() /
		((df_match["Average Mz_1"] + df_match["Average Mz_2"]) / 2)
	) * 1e6

	df_match["rt_error"] = (df_match["Average Rt_1"] - df_match["Average Rt_2"]).abs()
	df_match

	df_match_filter = df_match[df_match["mz_error"] < tol_mz]
	# df_match_filter = df_match[(df_match["mz_error"] < tol_mz) & (df_match["rt_error"] < tol_rt)]
	
	print(train_set)
	display(df_match_filter)

In [ ]:
tol_rt, tol_mz = 0.5, 15 # 0.5 min, 10 ppm

df_1 = pd.read_csv(f"data/{dataset1}/raw.csv", index_col=0).iloc[:, :3] # Sep
df_2 = pd.read_csv(f"data/{dataset2}/raw.csv", index_col=0).iloc[:, :3] # Dec
print(len(df_1), len(df_2))

df_match = (
	df_1.assign(key=1)
		.merge(df_2.assign(key=1), on="key", suffixes=("_1", "_2"))
		.drop(columns="key")
)

df_match["mz_error"] = (
	(df_match["Average Mz_1"] - df_match["Average Mz_2"]).abs() /
	((df_match["Average Mz_1"] + df_match["Average Mz_2"]) / 2)
) * 1e6

df_match["rt_error"] = (df_match["Average Rt_1"] - df_match["Average Rt_2"]).abs()
df_match

In [ ]:
df_match_filter = df_match[df_match["mz_error"] < tol_mz]
df_match_filter

In [ ]:
df_match_filter = df_match[(df_match["mz_error"] < tol_mz) & (df_match["rt_error"] < tol_rt)]
df_match_filter

In [ ]:
""" rt_diff = (df_match["Average Rt_1"] - df_match["Average Rt_2"]).abs()
mz_diff = (
	((df_match["Average Mz_1"] - df_match["Average Mz_2"]).abs() / (df_match["Average Mz_1"] + df_match["Average Mz_2"]) / 2)
) * 1e6
mask = (rt_diff <= tol_rt) & (mz_diff <= tol_mz)
# mask = (mz_diff <= tol_mz)

df_intersection = df_match.loc[mask].reset_index(drop=True)
df_intersection """

In [ ]:
""" tol_rt, tol_mz = 0.5, 0.01 # Change

for train_set in list_train_set:
	df_1 = pd.read_csv(f"data/{dataset1}/output/node_alignment_{encoder}_{dataset1}_{train_set}.csv" , sep=";", decimal=",")
	df_2 = pd.read_csv(f"data/{dataset2}/output/node_alignment_{encoder}_{dataset2}_{train_set}.csv" , sep=";", decimal=",")

	df_match = (
		df_1.assign(key=1)
			.merge(df_2.assign(key=1), on="key", suffixes=("_1", "_2"))
			.drop(columns="key")
	)

	rt_diff = (df_match["Average Rt_1"] - df_match["Average Rt_2"]).abs()
	mz_diff = (
		(df_match["Average Mz_1"] - df_match["Average Mz_2"]).abs() # / df_match["Average Mz_2"] * 1e6
	)
	# mask = (rt_diff <= tol_rt) & (mz_diff <= tol_mz)
	mask = (mz_diff <= tol_mz)

	df_intersection = df_match.loc[mask].reset_index(drop=True)
	print(train_set)
	display(df_intersection) """

### Comparison filter Deybis

In [ ]:
dataset1 = "deybis_filter_september_2br_10ar" # "deybis_filter_september_05", "deybis_filter_december_05", "deybis_05", "vanessa_05", "mentos_05", "Douban Online_Offline", "ACM_DBLP" # args.dataset
dataset2 = "deybis_filter_december_2br_10ar" # "deybis_filter_september_05", "deybis_filter_december_05",

# Parameters data
file = open(f"data/{dataset1}/input/parameters.json")
params = json.load(file)

methods = params["methods"]
print("Methods:\t", methods)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

has_transformation = params["has_transformation"]
print("Has transformation:", has_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

threshold_corr = params["threshold_corr"]
print("Threshold corr:\t", threshold_corr)

iterations = params["iterations"]
print("Iterations:\t", iterations)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups_id_no = params["groups_id_no"]
print("Groups id (no):\t", groups_id_no)

seeds = params["seeds"]
print("Seeds:\t\t", seeds)

sensitivity = params["sensitivity"]
print("Sensitivity:\t", sensitivity)

In [ ]:
# Read raw data

df_join_raw = pd.read_csv("data/{}/input/raw.csv".format(dataset1), index_col=0)
df_join_raw

In [ ]:
# Filter

list_node_intersection = []
for group_id in groups_id:
	print(group_id)

	# Read common node
	common_node_id1 = sorted(np.load(f"data/{dataset1}/output/common_node/{group_id}.npy"))
	common_node_id2 = sorted(np.load(f"data/{dataset2}/output/common_node/{group_id}.npy"))
	# print(common_node_id1)
	# print(common_node_id2)

	intersection = sorted(list(set(common_node_id1) & set(common_node_id2)))
	# print(intersection)
	
	list_node_intersection.append([group_id, intersection])
	# list_node_intersection.append([group_id, ",".join(map(str, intersection))])

	# Plot
	plt.figure()
	venn2(subsets=(len(common_node_id1), len(common_node_id2), len(intersection)), set_labels=("September", "December"))
	plt.show()

df_comparison = pd.DataFrame(list_node_intersection, columns=["Group id", "Common node id"])
df_comparison

In [ ]:
# Filter by gropu id

for i, group_id in enumerate(groups_id):
	print(group_id)
	common_node_id = list_node_intersection[i][1]
	print(len(common_node_id), common_node_id)

	# Filter raw data
	df_join_raw_filter = df_join_raw.loc[common_node_id].iloc[:, [0, 1, 2]]
	# df_join_raw_filter.to_csv(f"data/{dataset}/output/filter_raw/{group_id}.csv", sep=";", decimal=",", index_label="Id")
	display(df_join_raw_filter)

In [ ]:
# Comparison with Statistics

df_sta = pd.read_csv(f"data/deybis_filter_september_2br_3ar/input/sta.csv") # Prof. Alfredo
df_sta

In [ ]:
# Filter by gropu id (using intersection nodes)

tol_rt, tol_mz = 0.5, 0.01
for i, group_id in enumerate(groups_id):
	print(group_id)
	common_node_id = list_node_intersection[i][1]
	print(common_node_id)

	# Filter raw data
	df_join_raw_filter = df_join_raw.loc[common_node_id].iloc[:, [0, 1]]

	df_1 = df_sta.copy()
	df_2 = df_join_raw_filter.copy()
	print(len(df_1), len(df_2))

	df_match = (
		df_1.assign(key=1)
			.merge(df_2.assign(key=1), on="key", suffixes=("_1", "_2"))
			.drop(columns="key")
	)

	df_match["mz_error"] = (
		(df_match["Average Mz_1"] - df_match["Average Mz_2"]).abs()
	)

	df_match["rt_error"] = (df_match["Average Rt_1"] - df_match["Average Rt_2"]).abs()
	# df_match_filter = df_match[df_match["mz_error"] < tol_mz]
	df_match_filter = df_match[(df_match["mz_error"] < tol_mz) & (df_match["rt_error"] < tol_rt)]
	display(df_match_filter)

In [ ]:
# Filter by gropu id (using common nodes)

dataset = dataset2 # Change, dataset1, dataset2
tol_rt, tol_mz = 0.5, 0.01
print(dataset)

for i, group_id in enumerate(groups_id):
	print(group_id)
	common_node_id = sorted(np.load(f"data/{dataset}/output/common_node/{group_id}.npy"))
	print(common_node_id)

	# Filter raw data
	df_join_raw_filter = df_join_raw.loc[common_node_id].iloc[:, [0, 1]]

	df_1 = df_sta.copy()
	df_2 = df_join_raw_filter.copy()
	print(len(df_1), len(df_2))

	df_match = (
		df_1.assign(key=1)
			.merge(df_2.assign(key=1), on="key", suffixes=("_1", "_2"))
			.drop(columns="key")
	)

	df_match["mz_error"] = (
		(df_match["Average Mz_1"] - df_match["Average Mz_2"]).abs()
	)

	df_match["rt_error"] = (df_match["Average Rt_1"] - df_match["Average Rt_2"]).abs()
	# df_match_filter = df_match[df_match["mz_error"] < tol_mz]
	df_match_filter = df_match[(df_match["mz_error"] < tol_mz) & (df_match["rt_error"] < tol_rt)]
	display(df_match_filter)

### Classification

In [ ]:
# file = open("exp.json")
# experiment = json.load(file)
exp = "exp8" # experiment["exp"] # Change to static

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

methods = params["methods"]
print("Methods:\t", methods)

has_transformation = params["has_transformation"]
print("Has transformation:", has_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

cuda = params["cuda"]
print("Cuda:\t", cuda)

In [ ]:
# df_join_raw = pd.read_csv(f"experiments/input/{exp}_raw.csv", index_col=0)
# df_raw = pd.read_csv(f"experiments/raw_data/deybis_filter_september_all_format.csv", delimiter="|")
df_raw = pd.read_csv(f"experiments/raw_data/vanessa_december_all_format.csv", delimiter="|")
df_raw

In [ ]:
# Concat
df_join_raw = pd.concat([df_raw.iloc[:, :]], axis=1)
df_join_raw.set_index("Alignment ID", inplace=True)
df_join_raw = df_join_raw.rename_axis(None)
df_join_raw

In [ ]:
models = {
	# Linear models
	"LogisticRegression": LogisticRegression(random_state=42, max_iter=1000),
	"RidgeClassifier": RidgeClassifier(random_state=42),
	"SGDClassifier": SGDClassifier(random_state=42),
	"Perceptron": Perceptron(random_state=42),
	"PassiveAggressive": PassiveAggressiveClassifier(random_state=42),

	# Trees
	"DecisionTree": DecisionTreeClassifier(random_state=42),
	"ExtraTree": ExtraTreeClassifier(random_state=42),

	# Ensembles
	"RandomForest": RandomForestClassifier(random_state=42),
	"ExtraTrees": ExtraTreesClassifier(random_state=42),
	"Bagging": BaggingClassifier(random_state=42),
	"AdaBoost": AdaBoostClassifier(random_state=42),
	"GradientBoosting": GradientBoostingClassifier(random_state=42),
	"HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),

	# SVM
	"SVC": SVC(random_state=42),
	"NuSVC": NuSVC(random_state=42),
	"LinearSVC": LinearSVC(random_state=42),

	# KNN (no random_state parameter)
	"KNeighbors": KNeighborsClassifier(),
	"RadiusNeighbors": RadiusNeighborsClassifier(),

	# Neural network
	"MLPClassifier": MLPClassifier(random_state=42, max_iter=1000),

	# Naive Bayes (no random_state parameter)
	"GaussianNB": GaussianNB(),
	"MultinomialNB": MultinomialNB(),
	"BernoulliNB": BernoulliNB(),
	"ComplementNB": ComplementNB(),

	# Discriminant analysis (no random_state)
	"LDA": LinearDiscriminantAnalysis(),
	"QDA": QuadraticDiscriminantAnalysis(),

	# Gaussian Process (random_state available)
	"GaussianProcess": GaussianProcessClassifier(random_state=42)
}


In [ ]:
# Classification

print(raw_data_file)
list_metrics = []
for name, model in models.items():
	for group_id in groups_id + ["Intersection", "Union"]:
		try:
			# Read common node
			common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
			print(name, group_id, len(common_node_id))
			
			df_join_raw_signal = df_join_raw.loc[common_node_id].iloc[:, 3:]
			df_join_raw_signal_t = df_join_raw_signal.T
			# display(df_join_raw_signal_t)
			
			# Features
			X = df_join_raw_signal_t.values
			# print(X.shape)

			# Target
			y = df_join_raw_signal_t.index
			y = [item.split("_")[0] for item in y]
			# print(y)
					
			# Split
			X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
			# print(y_test)
			
			# Scaling
			scaler = StandardScaler()
			X_train = scaler.fit_transform(X_train)  # fit only on train
			X_test = scaler.transform(X_test)        # only transform on test
			
			# Select and train model
			model.fit(X_train, y_train)

			# Evaluation
			y_pred = model.predict(X_test)

			accuracy = accuracy_score(y_test, y_pred)
			recall_macro = recall_score(y_test, y_pred, average="macro")
			f1_macro = f1_score(y_test, y_pred, average="macro")
			# classification_rep = classification_report(y_test, y_pred, output_dict=True)
			
			cm = confusion_matrix(y_test, y_pred)
			class_names = model.classes_
			tick_marks = np.arange(len(class_names))
			
			""" plt.figure()
			plt.imshow(cm, cmap="Blues")
			plt.title("Confusion Matrix")
			# plt.colorbar()
			plt.xticks(tick_marks, class_names, rotation=45, ha="right", rotation_mode="anchor")
			plt.yticks(tick_marks, class_names)

			# Values inside each cell
			for i in range(cm.shape[0]):
				for j in range(cm.shape[1]):
					plt.text(
						j, i,
						format(cm[i, j], "d"),
						ha="center",
						va="center"
					)

			plt.xlabel("Predicted Label")
			plt.ylabel("True Label")
			plt.tight_layout()
			plt.show() """

			# Convert the dictionary to a pandas DataFrame and transpose it
			# df_report = pd.DataFrame(classification_rep).transpose()
			# display(df_report)
			list_metrics.append([name, group_id, len(common_node_id), accuracy, recall_macro, f1_macro])
		except:
			print(f"Error in {name} - {group_id}")

df_metrics = pd.DataFrame(list_metrics,  columns=["Model", "Group id", "Num. common node", "Accuracy", "Recall", "F1-score"])
df_metrics

In [ ]:
df_metrics.sort_values(by="Accuracy", ascending=False).head(50)

#### Select best model

In [ ]:
models

In [ ]:
# Select best model (dataset and group id)

exp = "exp8" # Change
group_id = "SecoCusco" # Change, Union, FrescoCusco
encoder = "GINE"
file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

# models_best = models # Change, models_best
models_best = {"RandomForest": RandomForestClassifier(random_state=42),} # {"ExtraTrees": ExtraTreesClassifier()}

In [ ]:
# September data

# Read common node
common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
print(group_id, len(common_node_id))

# df_join_raw_signal = df_join_raw.loc[common_node_id].iloc[:, 3:] # With common node
df_join_raw_signal = df_join_raw.iloc[:, 3:] # Without common node
df_join_raw_signal_t = df_join_raw_signal.T
# display(df_join_raw_signal_t)

# Features
X = df_join_raw_signal_t.values

# Target
y = df_join_raw_signal_t.index
y = [item.split("_")[0] for item in y]

# Count the occurrences of each item

counts = Counter(y)

# Extract labels (items) and heights (counts) for the plot
labels = list(counts.keys())
values = list(counts.values())

# Create the bar plot
plt.figure(figsize=(8, 5))
plt.bar(labels, values, color="skyblue")

plt.xticks(labels, labels, rotation=45, ha="right")
# Add labels and title
plt.xlabel("Classes")
plt.ylabel("Count")
plt.title("Instances")

# Display the plot
plt.show()

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # fit only on train
X_test = scaler.transform(X_test)        # only transform on test

In [ ]:
df_join_raw

In [ ]:
encoder

In [ ]:
# Read common node
common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
common_node_id

In [ ]:
# December data

# Evaluation with December data
# df_join_raw = pd.read_csv(f"experiments/input/{exp}_raw.csv", index_col=0)
# df_raw_new = pd.read_csv(f"experiments/raw_data/deybis_filter_december_all_format.csv", delimiter="|")
df_raw_new = pd.read_csv(f"experiments/raw_data/vanessa_december_all_format.csv", delimiter="|")


# Concat
df_join_raw_new = pd.concat([df_raw_new.iloc[:, :]], axis=1)
df_join_raw_new.set_index("Alignment ID", inplace=True)
df_join_raw_new = df_join_raw_new.rename_axis(None)
df_join_raw_new

# df_join_raw_signal_new = df_join_raw_new.loc[common_node_id].iloc[:, 3:] # With common node
df_join_raw_signal_new = df_join_raw_new.iloc[:, 3:] # Without common node

df_join_raw_signal_t_new = df_join_raw_signal_new.T
# display(df_join_raw_signal_t_)

# Features
X_new = df_join_raw_signal_t_new.values

# Target
y_new = df_join_raw_signal_t_new.index
y_new = [item.split("_")[0] for item in y_new]

# Count the occurrences of each item
counts = Counter(y_new)

# Extract labels (items) and heights (counts) for the plot
labels = list(counts.keys())
values = list(counts.values())

# Create the bar plot
plt.figure(figsize=(8, 5))
plt.bar(labels, values, color="skyblue")

plt.xticks(labels, labels, rotation=45, ha="right")
# Add labels and title
plt.xlabel("Classes")
plt.ylabel("Count")
plt.title("Instances")

# Display the plot
plt.show()

# Split
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size=0.1, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_new = scaler.fit_transform(X_train_new)  # fit only on train
X_test_new = scaler.transform(X_test_new)        # only transform on test """

X_test_new = scaler.fit_transform(X_new)        # only transform on test
y_test_new = y_new

In [ ]:
models_best

In [ ]:
map_group = {
	"FrescoAmazonas": "FreshAmazonas",
	"FrescoCusco": "FreshCusco",
	"FrescoSanMartin": "FreshSanMartín",
	"SecoAmazonas": "DryAmazonas",
	"SecoCusco": "DryCusco",
	"SecoSanMartin": "DrySanMartín",
}

In [ ]:
class_names

In [ ]:
# Iterations
list_metrics = []

for name, model in models_best.items():
	# Select and train model
	best_model = models[name]
	best_model.fit(X_train_new, y_train_new)
	
	# Evaluation
	y_pred_new = best_model.predict(X_test_new)

	accuracy = accuracy_score(y_test_new, y_pred_new)
	recall_macro = recall_score(y_test_new, y_pred_new, average="macro")
	f1_macro = f1_score(y_test_new, y_pred_new, average="macro")

	print("Train")
	print("Model:", name)
	print("Accuracy:", accuracy)
	print("Recall:", recall_macro)
	print("F1-score:", f1_macro)
	
	# Predict on December data
	y_pred_new = best_model.predict(X_test_new)
	# print(len(y_pred_new))
	
	accuracy = accuracy_score(y_test_new, y_pred_new)
	recall_macro = recall_score(y_test_new, y_pred_new, average="macro")
	f1_macro = f1_score(y_test_new, y_pred_new, average="macro")
	
	list_metrics.append([name, group_id, len(common_node_id), accuracy, recall_macro, f1_macro])
	
	""" print("Test")
	print("Model:", name)
	print("Accuracy:", accuracy)
	print("Recall:", recall_macro)
	print("F1-score:", f1_macro) """

	# Plot CM
	cm = confusion_matrix(y_test_new, y_pred_new)
	class_names = ['FreshAmazonas', 'FreshCusco', 'FreshSanMartin', 'DryAmazonas', 'DryCusco', 'DrySanMartin'] # best_model.classes_
	tick_marks = np.arange(len(class_names))

	plt.figure()
	plt.imshow(cm, cmap="Blues")
	plt.title("Confusion Matrix")
	# plt.colorbar()
	plt.xticks(tick_marks, class_names, rotation=45, ha="right", rotation_mode="anchor")
	plt.yticks(tick_marks, class_names)

	# Values inside each cell
	for i in range(cm.shape[0]):
		for j in range(cm.shape[1]):
			plt.text(
				j, i,
				format(cm[i, j], "d"),
				ha="center",
				va="center"
			)

	plt.xlabel("Predicted Label")
	plt.ylabel("True Label")
	plt.tight_layout()
	plt.show()

df_metrics = pd.DataFrame(list_metrics,  columns=["Model", "Group id", "Num. common node", "Accuracy", "Recall", "F1-score"])
df_metrics.sort_values(by="Accuracy", ascending=False)

In [ ]:
X_train_new.shape

Evaluation with December data

In [ ]:
""" df_comp = pd.DataFrame({
	"y true": y_test_new,
	"y pred": y_pred_new
})
df_comp.head(50) """

### Comparison T-GAE with Deybis proposal

In [ ]:
# file = open("exp.json")
# experiment = json.load(file)
exp = "exp1" # experiment["exp"] # Change to static

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

methods = params["methods"]
print("Methods:\t", methods)

has_transformation = params["has_transformation"]
print("Has transformation:", has_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

cuda = params["cuda"]
print("Cuda:\t", cuda)

In [ ]:
for group_id in groups_id + ["Intersection", "Union"]:
	# Read common node
	common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{group_id}.npy")
	print(group_id, len(common_node_id))
	print(common_node_id)

In [ ]:
# september min 82
filter_september = [0, 8, 25, 5, 16, 17, 15, 30, 4, 7, 29, 11, 6, 28, 68, 18, 31, 57, 12, 65, 14, 20, 9, 21, 75, 77, 10, 52, 13, 81, 19, 53, 26, 24, 58, 69, 2, 1, 35, 27, 64, 67, 45, 46, 78, 55, 71, 48, 76, 60, 23, 32, 42, 36, 3, 59, 73, 80, 63, 38, 72, 49, 56, 47, 22, 43, 61, 44, 41, 51, 66, 70, 74, 40, 79, 37, 34, 62, 50, 54, 33, 39]

# september 162
# filter_september = np.array([1, 54, 12, 158, 47, 53, 8, 5, 55, 30, 21, 29, 40, 49, 161, 6, 57, 61, 7, 31, 11, 9, 27, 51, 43, 28, 52, 16, 10, 134, 50, 143, 22, 122, 105, 153, 32, 140, 60, 157, 48, 41, 146, 58, 44, 23, 15, 34, 20, 125, 13, 133, 24, 124, 0, 145, 119, 144, 142, 17, 147, 18, 80, 135, 45, 26, 95, 66, 136, 123, 19, 159, 91, 35, 141, 154, 121, 33, 150, 139, 59, 112, 149, 37, 3, 107, 56, 128, 116, 38, 148, 2, 75, 126, 67, 103, 73, 155, 97, 113, 89, 110, 76, 151, 65, 96, 152, 132, 36, 25, 71, 85, 42, 120, 4, 39, 78, 72, 74, 104, 111, 90, 84, 14, 46, 130, 99, 62, 64, 160, 68, 77, 70, 137, 156, 82, 102, 108, 94, 87, 88, 86, 127, 114, 118, 138, 109, 79, 98, 129, 81, 115, 106, 63, 100, 83, 131, 69, 117, 92, 93, 101])

len(filter_september)

In [ ]:
best_features = 32  # Change 32 to 82 and 64 to 162
print(filter_september[:best_features])

In [ ]:
# September data

# df_join_raw = pd.read_csv(f"experiments/input/{exp}_raw.csv", index_col=0)
# df_raw = pd.read_csv(f"experiments/raw_data/deybis_filter_september_all_format.csv", delimiter="|")
df_raw = pd.read_csv(f"experiments/raw_data/deybis_filter_september_min_all_format.csv", delimiter="|")

df_raw

# Concat
df_join_raw = pd.concat([df_raw.iloc[:, :]], axis=1)
df_join_raw.set_index("Alignment ID", inplace=True)
df_join_raw = df_join_raw.rename_axis(None)
df_join_raw

In [ ]:
# December data

# Evaluation with December data
# df_join_raw = pd.read_csv(f"experiments/input/{exp}_raw.csv", index_col=0)
# df_raw_new = pd.read_csv(f"experiments/raw_data/deybis_filter_december_all_format.csv", delimiter="|")
df_raw_new = pd.read_csv(f"experiments/raw_data/deybis_filter_december_min_all_format.csv", delimiter="|")

# Concat
df_join_raw_new = pd.concat([df_raw_new.iloc[:, :]], axis=1)
df_join_raw_new.set_index("Alignment ID", inplace=True)
df_join_raw_new = df_join_raw_new.rename_axis(None)
df_join_raw_new

In [ ]:
models_best = {"RandomForest": RandomForestClassifier(random_state=42),} # {"ExtraTrees": ExtraTreesClassifier()}

In [ ]:
# Comparison with Statistics

common_node_id1 = sorted(filter_september[:best_features])
print(common_node_id1)

list_node_intersection = []
list_metrics = [] # september
list_metrics_new = [] # december

dict_common_node = {}

for group_id in groups_id + ["Intersection", "Union"]:
	print(group_id)

	# Read common node
	common_node_id2 = sorted(np.load(f"experiments/output/{exp}/common_nodes/{group_id}.npy"))
	# print(common_node_id1)
	# print(common_node_id2)

	intersection = sorted(list(set(common_node_id1) & set(common_node_id2)))
	# print(intersection)
	
	list_node_intersection.append([group_id, intersection])
	# list_node_intersection.append([group_id, ",".join(map(str, intersection))])

	# Plot Venn
	plt.figure()
	venn2(subsets=(len(common_node_id1), len(common_node_id2), len(intersection)), set_labels=("Deybis proposal", f"T-GAE ({group_id})"))
	plt.show()

	# Classification
	# Preprocessing
	# September data

	# Read common node
	common_node_id = intersection.copy() # np.load(f"experiments/output/{exp}/common_nodes/{group_id}.npy")
	print(group_id, len(common_node_id))
	dict_common_node[group_id] = common_node_id

	# September data
	df_join_raw_signal = df_join_raw.loc[common_node_id].iloc[:, 3:] # With common node
	# df_join_raw_signal = df_join_raw.iloc[:, 3:] # Without common node
	df_join_raw_signal_t = df_join_raw_signal.T
	# display(df_join_raw_signal_t)

	# Features
	X = df_join_raw_signal_t.values

	# Target
	y = df_join_raw_signal_t.index
	y = [item.split("_")[0] for item in y]

	# Count the occurrences of each item
	""" counts = Counter(y)

	# Extract labels (items) and heights (counts) for the plot
	labels = list(counts.keys())
	values = list(counts.values())

	# Create the bar plot
	plt.figure(figsize=(8, 5))
	plt.bar(labels, values, color="skyblue")

	plt.xticks(labels, labels, rotation=45, ha="right")
	# Add labels and title
	plt.xlabel("Classes")
	plt.ylabel("Count")
	plt.title("Instances")

	# Display the plot
	plt.show() """

	# Split
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

	# Scaling
	scaler = StandardScaler()
	X_train = scaler.fit_transform(X_train)  # fit only on train
	X_test = scaler.transform(X_test)        # only transform on test

	# Train
	for name, model in models_best.items():
		try:
			# Select and train model
			print(name)
			best_model = models_best[name]
			
			best_model.fit(X_train, y_train)

			# Evaluation
			y_pred = best_model.predict(X_test)

			accuracy = accuracy_score(y_test, y_pred)
			recall_macro = recall_score(y_test, y_pred, average="macro")
			f1_macro = f1_score(y_test, y_pred, average="macro")

			print("Train")
			print("Model:", name)
			print("Accuracy:", accuracy)
			print("Recall:", recall_macro)
			print("F1-score:", f1_macro)
			
			list_metrics.append([name, group_id, len(common_node_id), accuracy, recall_macro, f1_macro])
			
			""" print("Test")
			print("Model:", name)
			print("Accuracy:", accuracy)
			print("Recall:", recall_macro)
			print("F1-score:", f1_macro) """

			# Plot CM
			cm = confusion_matrix(y_test, y_pred)
			class_names = best_model.classes_
			tick_marks = np.arange(len(class_names))

			plt.figure()
			plt.imshow(cm, cmap="Blues")
			plt.title("Confusion Matrix")
			# plt.colorbar()
			plt.xticks(tick_marks, class_names, rotation=45, ha="right", rotation_mode="anchor")
			plt.yticks(tick_marks, class_names)

			# Values inside each cell
			for i in range(cm.shape[0]):
				for j in range(cm.shape[1]):
					plt.text(
						j, i,
						format(cm[i, j], "d"),
						ha="center",
						va="center"
					)

			plt.xlabel("Predicted Label")
			plt.ylabel("True Label")
			plt.tight_layout()
			plt.show()
		except Exception as e:
			print(f"Error in {name}: {e}")
			pass
	# Predict on December data
	df_join_raw_signal_new = df_join_raw_new.loc[common_node_id].iloc[:, 3:] # With common node
	df_join_raw_signal_t_new = df_join_raw_signal_new.T
	# display(df_join_raw_signal_t_new)

	# Features
	X_new = df_join_raw_signal_t_new.values

	# Target
	y_new = df_join_raw_signal_t_new.index
	y_new = [item.split("_")[0] for item in y_new]

	# Scaling
	scaler = StandardScaler()
	""" X_train_new = scaler.fit_transform(X_train_new)  # fit only on train
	X_test_new = scaler.transform(X_test_new)        # only transform on test """

	X_test_new = scaler.fit_transform(X_new)        # only transform on test
	y_test_new = y_new

	# Predict on December data
	y_pred_new = best_model.predict(X_test_new)
	# print(len(y_pred_new))
	
	accuracy = accuracy_score(y_test_new, y_pred_new)
	recall_macro = recall_score(y_test_new, y_pred_new, average="macro")
	f1_macro = f1_score(y_test_new, y_pred_new, average="macro")
	
	list_metrics_new.append([name, group_id, len(common_node_id), accuracy, recall_macro, f1_macro])

	# Plot CM
	cm = confusion_matrix(y_test_new, y_pred_new)
	class_names = best_model.classes_
	tick_marks = np.arange(len(class_names))

	plt.figure()
	plt.imshow(cm, cmap="Blues")
	plt.title("Confusion Matrix")
	# plt.colorbar()
	plt.xticks(tick_marks, class_names, rotation=45, ha="right", rotation_mode="anchor")
	plt.yticks(tick_marks, class_names)

	# Values inside each cell
	for i in range(cm.shape[0]):
		for j in range(cm.shape[1]):
			plt.text(
				j, i,
				format(cm[i, j], "d"),
				ha="center",
				va="center"
			)

	plt.xlabel("Predicted Label")
	plt.ylabel("True Label")
	plt.tight_layout()
	plt.show()

print("Metrics for September test data:")
df_metrics = pd.DataFrame(list_metrics,  columns=["Model", "Group id", "Num. common node", "Accuracy", "Recall", "F1-score"])
df_metrics.sort_values(by="Accuracy", ascending=False)
display(df_metrics)

print("Metrics for December data:")
df_metrics_new = pd.DataFrame(list_metrics_new,  columns=["Model", "Group id", "Num. common node", "Accuracy", "Recall", "F1-score"])
df_metrics_new.sort_values(by="Accuracy", ascending=False)
display(df_metrics_new)

df_comparison = pd.DataFrame(list_node_intersection, columns=["Group id", "Common node id"])
display(df_comparison)

print(dict_common_node)

In [ ]:
# New intersection

dict_common_node
print(dict_common_node.keys())

In [ ]:
seco_amazonas = set(dict_common_node["SecoAmazonas"])
seco_cusco = set(dict_common_node["SecoCusco"])
seco_san_martin = set(dict_common_node["SecoSanMartin"])

fresco_amazonas = set(dict_common_node["FrescoAmazonas"])
fresco_cusco = set(dict_common_node["FrescoCusco"])
fresco_san_martin = set(dict_common_node["FrescoSanMartin"])

dict_intersection = {
	"Seco": list(seco_amazonas & seco_cusco & seco_san_martin),
	"Fresco": list(fresco_amazonas & fresco_cusco & fresco_san_martin),
	"Amazonas": list(seco_amazonas & fresco_amazonas),
	"Cusco": list(seco_cusco & fresco_cusco),
	"San Martin": list(seco_san_martin & fresco_san_martin)
}
dict_intersection

In [ ]:
# Comparison with Statistics

list_node_intersection = []
list_metrics = [] # september
list_metrics_new = [] # december

for group_id, common_node_id in dict_intersection.items():
	print(group_id, len(common_node_id))

	# September data
	df_join_raw_signal = df_join_raw.loc[common_node_id].iloc[:, 3:] # With common node
	# df_join_raw_signal = df_join_raw.iloc[:, 3:] # Without common node
	df_join_raw_signal_t = df_join_raw_signal.T
	# display(df_join_raw_signal_t)

	# Features
	X = df_join_raw_signal_t.values

	# Target
	y = df_join_raw_signal_t.index
	y = [item.split("_")[0] for item in y]

	# Count the occurrences of each item
	""" counts = Counter(y)

	# Extract labels (items) and heights (counts) for the plot
	labels = list(counts.keys())
	values = list(counts.values())

	# Create the bar plot
	plt.figure(figsize=(8, 5))
	plt.bar(labels, values, color="skyblue")

	plt.xticks(labels, labels, rotation=45, ha="right")
	# Add labels and title
	plt.xlabel("Classes")
	plt.ylabel("Count")
	plt.title("Instances")

	# Display the plot
	plt.show() """

	# Split
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

	# Scaling
	scaler = StandardScaler()
	X_train = scaler.fit_transform(X_train)  # fit only on train
	X_test = scaler.transform(X_test)        # only transform on test

	# Train
	for name, model in models_best.items():
		try:
			# Select and train model
			print(name)
			best_model = models_best[name]
			
			best_model.fit(X_train, y_train)

			# Evaluation
			y_pred = best_model.predict(X_test)

			accuracy = accuracy_score(y_test, y_pred)
			recall_macro = recall_score(y_test, y_pred, average="macro")
			f1_macro = f1_score(y_test, y_pred, average="macro")

			print("Train")
			print("Model:", name)
			print("Accuracy:", accuracy)
			print("Recall:", recall_macro)
			print("F1-score:", f1_macro)
			
			list_metrics.append([name, group_id, len(common_node_id), accuracy, recall_macro, f1_macro])
			
			""" print("Test")
			print("Model:", name)
			print("Accuracy:", accuracy)
			print("Recall:", recall_macro)
			print("F1-score:", f1_macro) """

			# Plot CM
			cm = confusion_matrix(y_test, y_pred)
			class_names = best_model.classes_
			tick_marks = np.arange(len(class_names))

			plt.figure()
			plt.imshow(cm, cmap="Blues")
			plt.title("Confusion Matrix")
			# plt.colorbar()
			plt.xticks(tick_marks, class_names, rotation=45, ha="right", rotation_mode="anchor")
			plt.yticks(tick_marks, class_names)

			# Values inside each cell
			for i in range(cm.shape[0]):
				for j in range(cm.shape[1]):
					plt.text(
						j, i,
						format(cm[i, j], "d"),
						ha="center",
						va="center"
					)

			plt.xlabel("Predicted Label")
			plt.ylabel("True Label")
			plt.tight_layout()
			plt.show()
		except Exception as e:
			print(f"Error in {name}: {e}")
			pass
	# Predict on December data
	df_join_raw_signal_new = df_join_raw_new.loc[common_node_id].iloc[:, 3:] # With common node
	df_join_raw_signal_t_new = df_join_raw_signal_new.T
	# display(df_join_raw_signal_t_new)

	# Features
	X_new = df_join_raw_signal_t_new.values

	# Target
	y_new = df_join_raw_signal_t_new.index
	y_new = [item.split("_")[0] for item in y_new]

	# Scaling
	scaler = StandardScaler()
	""" X_train_new = scaler.fit_transform(X_train_new)  # fit only on train
	X_test_new = scaler.transform(X_test_new)        # only transform on test """

	X_test_new = scaler.fit_transform(X_new)        # only transform on test
	y_test_new = y_new

	# Predict on December data
	y_pred_new = best_model.predict(X_test_new)
	# print(len(y_pred_new))
	
	accuracy = accuracy_score(y_test_new, y_pred_new)
	recall_macro = recall_score(y_test_new, y_pred_new, average="macro")
	f1_macro = f1_score(y_test_new, y_pred_new, average="macro")
	
	list_metrics_new.append([name, group_id, len(common_node_id), accuracy, recall_macro, f1_macro])

	# Plot CM
	cm = confusion_matrix(y_test_new, y_pred_new)
	class_names = best_model.classes_
	tick_marks = np.arange(len(class_names))

	plt.figure()
	plt.imshow(cm, cmap="Blues")
	plt.title("Confusion Matrix")
	# plt.colorbar()
	plt.xticks(tick_marks, class_names, rotation=45, ha="right", rotation_mode="anchor")
	plt.yticks(tick_marks, class_names)

	# Values inside each cell
	for i in range(cm.shape[0]):
		for j in range(cm.shape[1]):
			plt.text(
				j, i,
				format(cm[i, j], "d"),
				ha="center",
				va="center"
			)

	plt.xlabel("Predicted Label")
	plt.ylabel("True Label")
	plt.tight_layout()
	plt.show()

print("Metrics for September test data:")
df_metrics = pd.DataFrame(list_metrics,  columns=["Model", "Group id", "Num. common node", "Accuracy", "Recall", "F1-score"])
df_metrics.sort_values(by="Accuracy", ascending=False)
display(df_metrics)

print("Metrics for December data:")
df_metrics_new = pd.DataFrame(list_metrics_new,  columns=["Model", "Group id", "Num. common node", "Accuracy", "Recall", "F1-score"])
df_metrics_new.sort_values(by="Accuracy", ascending=False)
display(df_metrics_new)

df_comparison = pd.DataFrame(list_node_intersection, columns=["Group id", "Common node id"])
display(df_comparison)

print(dict_common_node)

### Intersection Cacao (september-december)

In [ ]:
# file = open("exp.json")
# experiment = json.load(file)
exp = "exp2" # experiment["exp"] # Change to static

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

methods = params["methods"]
print("Methods:\t", methods)

has_transformation = params["has_transformation"]
print("Has transformation:", has_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

cuda = params["cuda"]
print("Cuda:\t", cuda)

In [ ]:
for group_id in groups_id + ["Intersection", "Union"]:
	# Read common node
	common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{group_id}.npy")
	print(group_id, len(common_node_id))
	print(common_node_id)